In [ ]:
import pandas as pd

In [ ]:
df=pd.read_csv('/content/tweets[2].csv')

In [ ]:
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from string import punctuation
import nltk
import string
from nltk.stem import WordNetLemmatizer # Import WordNetLemmatizer
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet') # Download the wordnet corpus
nltk.download('punkt_tab') # Download punkt_tab
lemmatizer = WordNetLemmatizer() # Initialize the lemmatizer
stop_words = set(stopwords.words('english'))
def preprocess_text(text):
    text=text.lower()
    text=text.translate(str.maketrans('', '', string.punctuation)) # Remove punctuation
    text=re.sub(r'[^a-zA-Z0-9\s]', '', text)
    tokens=word_tokenize(text)#tokenization
    stop_words=set(stopwords.words('english'))#stop words
    tokens=[word for word in tokens if word not in stop_words]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]  # Lemmatization
    return " ".join(tokens)
df["cleaned_text"]=df["text"].astype(str).apply(preprocess_text)
print(df)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


          id  keyword                 location  \
0          0   ablaze                      NaN   
1          1   ablaze                      NaN   
2          2   ablaze            New York City   
3          3   ablaze           Morgantown, WV   
4          4   ablaze                      NaN   
...      ...      ...                      ...   
11365  11365  wrecked  Blue State in a red sea   
11366  11366  wrecked               arohaonces   
11367  11367  wrecked                       🇵🇭   
11368  11368  wrecked           auroraborealis   
11369  11369  wrecked                      NaN   

                                                    text  target  \
0      Communal violence in Bhainsa, Telangana. "Ston...       1   
1      Telangana: Section 144 has been imposed in Bha...       1   
2      Arsonist sets cars ablaze at dealership https:...       1   
3      Arsonist sets cars ablaze at dealership https:...       1   
4      "Lord Jesus, your love brings freedom and pard...   

In [ ]:
!pip install gensim

In [ ]:
import numpy as np
from gensim.models import Word2Vec
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
tokenized_X=[sentence.split() for sentence in df["cleaned_text"]]

In [ ]:
w2v_model=Word2Vec(
    sentences=tokenized_X,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)

In [ ]:
word_index={word:idx+1 for idx,word in enumerate(w2v_model.wv.index_to_key)}
sequences=[]
for sentences in tokenized_X:
  seq=[word_index[word] for word in sentences if word in word_index]
  sequences.append(seq)
print("Sequences:",sequences)

Sequences: [[4397, 702, 1574, 1656, 1029, 4670, 340, 70, 70, 470, 167, 871], [1656, 2168, 12155, 6585, 1574, 217, 502, 380, 2328, 1934, 34, 425, 217, 738, 1357], [1160, 167, 195, 871, 6576, 12154], [1160, 167, 195, 871, 6576, 12153, 12152], [1169, 811, 60, 1885, 1346, 12151, 3131, 1576, 3137, 167, 108, 871, 595, 12150], [129, 781, 349, 16, 762, 2397, 1120, 197, 16, 871, 12149, 16, 113, 1327, 1229], [538, 70, 167, 871, 12148, 1157, 12147, 2129, 4645, 382, 661, 788, 2639, 12146], [12145, 1646, 993, 12144, 1157, 167, 871, 51, 140, 1646, 5827, 6491, 553, 476, 69, 136], [329, 606, 726, 5823, 12142, 445, 5863, 167, 936, 871, 761, 670, 105, 12129], [2638, 979, 860, 885, 12141, 1024, 871, 78, 2643, 1896, 1896, 12140], [773, 1047, 3703, 369, 2639, 168, 6542, 70, 6504, 4651, 168, 667], [1120, 197, 328, 4649, 12139, 12138, 2649, 676, 1041, 248, 743, 12137, 12136], [12135, 1688, 167, 121, 993, 12134, 374, 1309, 1945, 121, 871, 6844, 4667, 6851, 12133, 121, 993], [12132, 799, 702, 1, 12131, 758, 58

In [ ]:
#padding sequences
padded_sequences=pad_sequences(sequences,padding='post')
print("Padded Sequences:\n",padded_sequences)

Padded Sequences:
 [[ 4397   702  1574 ...     0     0     0]
 [ 1656  2168 12155 ...     0     0     0]
 [ 1160   167   195 ...     0     0     0]
 ...
 [  101  1683   308 ...     0     0     0]
 [  439   148  8953 ...     0     0     0]
 [18528 23176   795 ...     0     0     0]]


In [ ]:
import numpy as np

def get_avg_vector(tokens, model, vector_size):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(vector_size)  # if no known word
    return np.mean(vectors, axis=0)

X_vectors = np.array([get_avg_vector(tokens, w2v_model, 100) for tokens in tokenized_X])
print("Shape of feature matrix:", X_vectors.shape)


Shape of feature matrix: (11370, 100)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_vectors, df["target"], test_size=0.2, random_state=42)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "SVM": SVC(kernel="linear")
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"\n{name} Results:")
    print("Train Accuracy:", model.score(X_train, y_train))
    print("Test Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))



Logistic Regression Results:
Train Accuracy: 0.8111257695690414
Test Accuracy: 0.8258575197889182
              precision    recall  f1-score   support

           0       0.83      1.00      0.90      1878
           1       0.00      0.00      0.00       396

    accuracy                           0.83      2274
   macro avg       0.41      0.50      0.45      2274
weighted avg       0.68      0.83      0.75      2274



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



Random Forest Results:
Train Accuracy: 0.9998900615655233
Test Accuracy: 0.8474054529463501
              precision    recall  f1-score   support

           0       0.85      0.99      0.91      1878
           1       0.84      0.15      0.26       396

    accuracy                           0.85      2274
   macro avg       0.84      0.57      0.59      2274
weighted avg       0.85      0.85      0.80      2274


SVM Results:
Train Accuracy: 0.8111257695690414
Test Accuracy: 0.8258575197889182
              precision    recall  f1-score   support

           0       0.83      1.00      0.90      1878
           1       0.00      0.00      0.00       396

    accuracy                           0.83      2274
   macro avg       0.41      0.50      0.45      2274
weighted avg       0.68      0.83      0.75      2274



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


*   Logistic Regression & SVM: High accuracy (~82%) but completely failed to detect the minority class (Class 1 → precision/recall/F1 = 0).

*   Random Forest: Best performer → highest test accuracy (~85%) and non-zero precision (0.84), recall (0.15), and F1 (0.26) for Class 1.

*   Random Forest works best with Word2Vec embeddings since it can model non-linear boundaries and at least identifies some positive cases.



